# Detecção Hierárquica de Toxicidade no Civil Comments
## CNN + Bi-LSTM com seleção *nested* de thresholds

**Artefato computacional final da iniciação científica**

Este notebook documenta a metodologia e preserva a reprodução final do experimento hierárquico no dataset **Civil Comments**.

| Execução | End-to-end Macro F1 |
|---|---:|
| Benchmark principal documentado | **0.4412** |
| Repetição anterior | **0.4427** |
| **Reprodução final preservada aqui** | **0.4423** |

Os três resultados sustentam uma região de desempenho estável em torno de **0.44 Macro F1**. O benchmark oficial permanece `0.4412` para evitar *cherry-picking*.

## 1. Pergunta de pesquisa e hipótese

**Pergunta:** uma arquitetura híbrida CNN + Bi-LSTM, organizada em dois estágios e avaliada com seleção de thresholds somente na validação interna, consegue produzir uma classificação de toxicidade metodologicamente válida no Civil Comments?

**Hipótese:** em uma tarefa hierárquica e desbalanceada, thresholds fixos podem estar desalinhados com a distribuição das probabilidades. Selecionar os pontos de decisão apenas na validação interna deve melhorar a Macro F1 end-to-end sem alterar o encoder.

## 2. Dataset, targets e arquitetura

**Dataset:** `google/civil_comments` — split de treino com **1.804.874 comentários**.

**Stage 1**
- `toxicity` — sinal principal de roteamento;
- `severe_toxicity` — saída auxiliar;
- verdade de referência para roteamento: `toxicity >= 0.4`.

**Stage 2**
- `obscene`
- `threat`
- `insult`
- `identity_attack`
- `sexual_explicit`

Os targets são preservados como scores fracionários durante o treinamento.

**Encoder:** embedding treinável → ramo Bi-LSTM + ramos Conv1D (kernels 2, 3 e 4) → concatenação → dropout → Dense → saídas sigmoid.

## 3. Protocolo experimental

O experimento utiliza **2 folds externos**. Em cada fold:

1. o fold externo é reservado exclusivamente para avaliação;
2. o restante é dividido em treino e validação interna;
3. o tokenizer é ajustado somente nos dados de treino;
4. `EarlyStopping` usa somente a validação interna;
5. thresholds do Stage 2 são selecionados na validação interna roteada;
6. o threshold do Stage 1 é selecionado pela Macro F1 end-to-end da validação interna;
7. todos os thresholds são congelados antes da avaliação externa.

A métrica principal é a **Macro F1 end-to-end**, pois incorpora a propagação de erro entre os dois estágios.

## 4. Preparação do ambiente

A reprodução preservada foi executada a partir do commit `cfcebb2` da `main`. O TensorFlow não encontrou CUDA nesse runtime; portanto, o full run foi executado em **CPU**.

O runner canônico é:

```bash
python scripts/run_hierarchical_cv.py --n-splits 2 --epochs 5
```

In [1]:
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/Umbura/Hatespeech_Detection_Civil_Comments_NLP_Obsolete.git"
REPO_DIR = Path("/content/Hatespeech_Detection_Civil_Comments_NLP_Obsolete")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "--prune"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "switch", "main"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
print("Commit:", subprocess.check_output(["git","log","-1","--oneline"], cwd=REPO_DIR, text=True).strip())

Repository: /content/Hatespeech_Detection_Civil_Comments_NLP_Obsolete
Commit: cfcebb2 Merge pull request #13 from Umbura/chore/remove-agents-md


In [2]:
%pip install -q   tensorflow==2.20.0   datasets==5.0.0   iterative-stratification==0.1.9   imbalanced-learn==0.14.2

## 5. Cobertura do gate

A análise abaixo confirma que `toxicity >= 0.4` preserva a grande maioria dos comentários que apresentam pelo menos um subtipo positivo.

In [3]:
!python scripts/analyze_gate_coverage.py

{
  "any_stage2_positive": {
    "missed_count": 533,
    "missed_rate": 0.004221782178217822,
    "positive_count": 126250,
    "routed_count": 125717
  },
  "gate_threshold": 0.4,
  "label_threshold": 0.5,
  "per_label": {
    "identity_attack": {"missed_count": 139, "missed_rate": 0.010365398956002983, "positive_count": 13410, "routed_count": 13271},
    "insult": {"missed_count": 136, "missed_rate": 0.0012765877560215518, "positive_count": 106534, "routed_count": 106398},
    "obscene": {"missed_count": 21, "missed_rate": 0.0021868166198063107, "positive_count": 9603, "routed_count": 9582},
    "sexual_explicit": {"missed_count": 226, "missed_rate": 0.04822876653862569, "positive_count": 4686, "routed_count": 4460},
    "threat": {"missed_count": 13, "missed_rate": 0.0030373831775700934, "positive_count": 4280, "routed_count": 4267}
  },
  "routed_samples": 201476,
  "total_samples": 1804874
}


## 6. Execução completa preservada

A saída abaixo é a consolidação científica do full run concluído. As barras de progresso do TensorFlow foram omitidas por legibilidade; fold sizes, melhores épocas, thresholds e métricas finais são preservados.

In [4]:
!python scripts/run_hierarchical_cv.py     --n-splits 2     --epochs 5

--- Ground-truth gate coverage analysis ---
Routed samples: 201476/1804874 using toxicity >= 0.40
obscene: positives=9603, missed=21 (0.22%)
threat: positives=4280, missed=13 (0.30%)
insult: positives=106534, missed=136 (0.13%)
identity_attack: positives=13410, missed=139 (1.04%)
sexual_explicit: positives=4686, missed=226 (4.82%)
any Stage 2 label: positives=126250, missed=533 (0.42%)

Fold 1/2
Stage 1 rows: fit=812193, inner_validation=90244, outer_eval=902437
Stage 2 rows: fit=90664, inner_validation=10074, outer_eval=100738
Stage 1 best inner-validation epoch: 2
Stage 2 best inner-validation epoch: 3
Selected inner thresholds: routing=0.28 (inner end-to-end Macro F1=0.4397, gate recall=0.5776, routing rate=0.0926); obscene=0.36, threat=0.42, insult=0.41, identity_attack=0.35, sexual_explicit=0.41

Fold 2/2
Stage 1 rows: fit=812193, inner_validation=90244, outer_eval=902437
Stage 2 rows: fit=90664, inner_validation=10074, outer_eval=100738
Stage 1 best inner-validation epoch: 2
Stag

## 7. Resultados da reprodução final

### Stage 1

| Métrica | Fixo | Nested |
|---|---:|---:|
| Precision | 0.8389 | 0.6962 |
| Recall | 0.4092 | **0.5879** |
| F1 | 0.5501 | **0.6375** |
| PR-AUC / AP | 0.7090 | 0.7090 |
| ROC-AUC | 0.9194 | 0.9194 |

### Stage 2 oracle

| Label | F1 fixo | F1 nested |
|---|---:|---:|
| obscene | 0.5280 | **0.6169** |
| threat | 0.4036 | **0.5114** |
| insult | 0.7485 | **0.7645** |
| identity_attack | 0.3311 | **0.5628** |
| sexual_explicit | 0.3844 | **0.5328** |
| **Macro F1** | **0.4791** | **0.5977** |

### End-to-end

| Label | F1 fixo | F1 nested |
|---|---:|---:|
| obscene | 0.4530 | **0.5112** |
| threat | 0.1870 | **0.3002** |
| insult | **0.6306** | 0.6291 |
| identity_attack | 0.1933 | **0.3809** |
| sexual_explicit | 0.2733 | **0.3900** |
| **Macro F1** | **0.3474** | **0.4423** |

Ganho end-to-end: **+0.0949 absoluto**, aproximadamente **+27.3% relativo**.

## 8. Interpretação

O resultado final confirma a hipótese experimental. PR-AUC e ROC-AUC do Stage 1 permaneceram altos (`0.7090` e `0.9194`), enquanto recall e F1 melhoraram fortemente após a seleção nested. Isso indica que parte relevante da degradação vinha do ponto de decisão, e não apenas da capacidade do encoder.

O Stage 2 oracle atingiu `0.5977`, enquanto o sistema end-to-end ficou em `0.4423`. O gap de `0.1554` mostra que o roteamento do Stage 1 continua sendo o principal ponto de propagação de erro.

A reprodução final também se manteve muito próxima do benchmark principal (`0.4412`) e da repetição anterior (`0.4427`), reforçando a estabilidade da região de desempenho.

## 9. Contexto externo

Benchmarks públicos no Civil Comments reportam Macro F1 em faixas próximas, embora com protocolos diferentes. Por isso, comparações com Transformers e outras arquiteturas devem ser tratadas como **contextualização**, não como comparação 1:1 ou alegação de superioridade.

O projeto não reivindica estado da arte. A contribuição é a formulação hierárquica, a correção metodológica contra leakage e a demonstração de que seleção nested de thresholds melhora significativamente o desempenho da cascata sem substituir o encoder.

## 10. Limitações

1. apenas 2 folds externos, devido ao custo computacional;
2. sinais de overfitting em épocas iniciais, mitigados por `EarlyStopping`;
3. propagação de erro do Stage 1 para o Stage 2;
4. `sexual_explicit` é o subtipo mais afetado pelo gate de verdade de referência;
5. fairness e robustez por subgrupos não foram avaliadas;
6. o benchmark é validação cruzada sobre o split de treino, não avaliação congelada no split oficial de teste;
7. pequenas variações podem ocorrer entre hardware e runtimes;
8. comparações externas não são 1:1;
9. não há alegação de prontidão para produção.

## 11. Conclusão

A seleção nested de thresholds elevou o Stage 1 F1 de `0.5501` para `0.6375`, o Stage 2 oracle Macro F1 de `0.4791` para `0.5977` e o sistema end-to-end de `0.3474` para **`0.4423` Macro F1**.

O resultado reproduz os experimentos anteriores (`0.4412` e `0.4427`) e sustenta a conclusão de que thresholds fixos estavam desalinhados com uma tarefa hierárquica e desbalanceada.

Para o escopo desta iniciação científica, o experimento é considerado concluído nesta configuração.

## 12. Referências

- Google / Jigsaw. **Civil Comments** dataset.
- Duchêne, C.; Jamet, H.; Guillaume, P.; Dehak, R. *A benchmark for toxic comment classification on Civil Comments dataset*. 2023.
- Rule, A. et al. *Ten simple rules for writing and sharing computational analyses in Jupyter Notebooks*. PLOS Computational Biology, 2019.
- Zhou, C. et al. *A C-LSTM Neural Network for Text Classification*. COLING, 2016.
- Schuster, M.; Paliwal, K. K. *Bidirectional recurrent neural networks*. IEEE Transactions on Signal Processing, 1997.

Documentação adicional:
- `docs/TARGET_STRATEGY.md`
- `docs/EXPERIMENT_HISTORY.md`
- `docs/REPRODUCIBILITY.md`
- `results/FINAL_RESULTS.md`
- `results/final_metrics.json`